# Controlled vessel-reference energy modelling
Default run: **SYNTHETIC EXAMPLE — NOT REAL RESULTS**. No competition downloads or submissions. These labels describe controlled added-reference energy, not natural vessel presence. Absolute export calibration is unverified. Set all three real paths below to use downloaded competition files; real inference is never evaluated against synthetic labels.
Run from the repository with `.venv/bin/jupyter nbconvert --to notebook --execute notebooks/solution.ipynb --output-dir outputs --ExecutePreprocessor.kernel_name=vessel-noise`. Register that kernel first with `.venv/bin/python -m ipykernel install --prefix .venv --name vessel-noise`.

In [ ]:
RELEASE_DIR = None
TEST_CSV = None
SAMPLE_SUB = None
SEED = 42
# Inspect the real DATA_DICTIONARY.md before changing quality filtering.
LOADER_OPTIONS = dict(drop_bad_flags=False)


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image
from src.synthetic import write_synthetic_release, make_synthetic_competition
from src.train import train
from src.predict import predict, COLUMNS
from src.evaluate import evaluate_frames
from src.pressure_map import pressure_map
from src.explain import explain
OUT = ROOT / 'outputs/notebook_run'
OUT.mkdir(parents=True, exist_ok=True)
SYNTHETIC = all(p is None for p in [RELEASE_DIR, TEST_CSV, SAMPLE_SUB])
if not SYNTHETIC and any(p is None for p in [RELEASE_DIR, TEST_CSV, SAMPLE_SUB]):
    raise ValueError('Set all three real paths, or leave all None for the synthetic example')
if SYNTHETIC:
    print('SYNTHETIC EXAMPLE — NOT REAL RESULTS')
    RELEASE_DIR = write_synthetic_release(OUT / 'SYNTHETIC_release', days=.25, seed=SEED)
    comp = make_synthetic_competition(OUT / 'SYNTHETIC_competition', n_test=60, seed=SEED + 4200)
    TEST_CSV, SAMPLE_SUB = comp / 'Test.csv', comp / 'sample_submission.csv'


## Train with site-held-out validation
Training labels use parametric **SYNTHETIC additions**, even when the backgrounds are real. Neither Test.csv nor sample placeholders are used for training, imputation, normalisation or calibration. Augmentations share a background-window identifier so repeated observations can be cluster-bootstrapped. Ensemble weights are selected on OOF predictions, so their OOF metric is a selection estimate; synthetic held-out sites provide a separate check.

In [ ]:
bundle, metrics = train(RELEASE_DIR, OUT / 'models', augment_k=3, seed=SEED, **LOADER_OPTIONS)
display(pd.DataFrame(metrics['oof']).T)


In [ ]:
submission = predict(TEST_CSV, bundle, SAMPLE_SUB, OUT / 'submission.csv')
sample = pd.read_csv(SAMPLE_SUB, dtype={'ID': str})
assert list(submission) == COLUMNS
assert submission.ID.tolist() == sample.ID.tolist()
assert np.isfinite(submission.iloc[:, 1:]).all().all()
assert submission.iloc[:, 1:].ge(0).all().all() and submission.iloc[:, 1:].le(1).all().all()
display(submission.head())


In [ ]:
if SYNTHETIC:
    labels = pd.read_csv(comp / 'SYNTHETIC_test_labels.csv')
    evaluation = evaluate_frames(submission, labels, seed=SEED)
    print('SYNTHETIC held-out evaluation:', json.dumps(evaluation, indent=2))
else:
    print('Real test: no labels. Evaluation skipped; do not infer labels from test inputs.')


In [ ]:
summary = pressure_map(RELEASE_DIR, OUT / 'maps', OUT / 'models/bundle.joblib', **LOADER_OPTIONS)
display(summary)
for filename in ['station_map.png', 'daily_relative.png', 'daily_inferred_share.png']:
    display(Image(filename=str(OUT / 'maps' / filename), width=800))


## Model explanations
Gain and standardised Ridge coefficients describe fitted predictors. TreeSHAP contributions are aggregated over training features, not test inputs; they are not causal explanations. Per-band aggregation includes explicitly band-indexed features only; global and band-group features remain in the family summary. Maps show total sound or **model-inferred** added-vessel-like share, not measured natural vessel fractions.

In [ ]:
features = pd.read_parquet(OUT / 'models/SYNTHETIC_addition_features.parquet')
explanations = explain(bundle, features, OUT / 'explanations', synthetic=SYNTHETIC)
display(explanations['contributions_by_family'])
display(Image(filename=str(OUT / 'explanations/contributions_by_family.png'), width=800))
display(Image(filename=str(OUT / 'explanations/contributions_by_band.png'), width=800))
